In [ ]:
import numpy as np
import matplotlib.pyplot as plt

V_rest = -65
V_thresh = -50
RC = 20
dt = 1
I = 16
ref = 0

V = np.ones(500) * V_rest

for t in range(1, len(V)):
    if ref > 0:
        ref = ref - 1
        continue
    V[t] = V[t-1] + dt/RC * (-(V[t-1] - V_rest) + I)
    if V[t] > V_thresh:
        V[t] = V_rest;
        ref = 5 # the neuron rests for 5ms before it can take any new inputs

plt.plot(V)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = 1
w = 300.0 # if neuron_0 is fired, then neuron_1 will receive an additional current w
w_list = [None] * 500
A_plus = 20
A_minus = 20
tau_stdp = 20.0

class Neuron:
    def __init__(self, V_rest, V_thresh, RC, I, ref, last_spike_time):
        self.V_rest = V_rest
        self.V_thresh = V_thresh
        self.RC = RC
        self.I = I
        self.ref = ref
        self.status = np.ones(500) * V_rest
        self.last_spike_time = last_spike_time

n_0 = Neuron(-65, -50, 20, 16, 0, -1000) # init last_spike_time to -1000 to represent this neuron has never fired at the beginning
n_1 = Neuron(-65, -50, 20, 0, 0, -1000)

pending_current = 0

for t in range(1, len(n_0.status)):
    
    # after n_0 is fired, n_1 receives an addtional current w in the next timestamp
    if pending_current > 0:
        n_1.I = pending_current
        pending_current = 0
    
    if n_0.ref > 0:
        n_0.ref -= 1
    else:
        n_0.status[t] = n_0.status[t-1] + dt/n_0.RC * (-(n_0.status[t-1] - n_0.V_rest) + n_0.I)
        if n_0.status[t] > n_0.V_thresh:
            n_0.last_spike_time = t # update its last_spike_time to t
            n_0.status[t] = n_0.V_rest
            n_0.ref = 5
            pending_current = w
            dt_stdp = n_1.last_spike_time - n_0.last_spike_time
            if dt_stdp < 0:  # post before pre
                w -= A_minus * np.exp(dt_stdp / tau_stdp)
            
    if n_1.ref > 0:
        n_1.ref -= 1
    else:
        n_1.status[t] = n_1.status[t-1] + dt/n_1.RC * (-(n_1.status[t-1] - n_1.V_rest) + n_1.I)
        n_1.I = 0
        if n_1.status[t] > n_1.V_thresh:
            print(f"n_1 fired at t={t}") # it's tricky to see it reflected on the graph since firing and resetting happen simultaneously  
            n_1.last_spike_time = t
            n_1.status[t] = n_1.V_rest
            n_1.ref = 5
            dt_stdp = n_1.last_spike_time - n_0.last_spike_time
            if dt_stdp > 0:  # pre before post
                w += A_plus * np.exp(-dt_stdp / tau_stdp)

    w_list[t] = w
    

# plt.plot(n_0.status)
# plt.plot(n_1.status)
# plt.plot(w_list)
fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.plot(n_0.status)
ax1.plot(n_1.status)
ax2.plot(w_list)
plt.show()

In [ ]:
# REF = 30ms, PREDICTION OCCURS!
import numpy as np
import matplotlib.pyplot as plt

T = 10000
stim = np.zeros((3, T))
# stim array pre-setting:
# we define every 60ms counts as a cycle
for cycle in range (T // 60):
    base = cycle * 60 + 1 # offset by 1 to coincide t iterating from 1
    if base < T: stim[0][base] = 400
    if base < 5000: # normal stim
        if base + 20 < T: stim[1][base + 20] = 400
        if base + 40 < T: stim[2][base + 40] = 400
    else: # removed stim for n_1 after 5000ms
        if base + 40 < T: stim[2][base + 40] = 400


w01 = 1.0 # if neuron_0 is fired, then neuron_1 will receive an additional input w01
w12 = 1.0 # if neuron_1 is fired, then neuron_2 will receive an additional input w12
w01_list = [None] * T
w12_list = [None] * T
n1_spike_times = []

dt = 1
A_plus = 30
A_minus = 20
tau_stdp = 20.0
n_0_fired = False
n_1_fired = False

class Neuron:
    def __init__(self, V_rest, V_thresh, RC, I, ref, last_spike_time):
        self.V_rest = V_rest
        self.V_thresh = V_thresh
        self.RC = RC
        self.I = I
        self.ref = ref
        self.status = np.ones(T) * V_rest
        self.last_spike_time = last_spike_time

n_0 = Neuron(-65, -50, 20, 0, 0, -1000) # init last_spike_time to -1000 to represent this neuron has never fired at the beginning
n_1 = Neuron(-65, -50, 20, 0, 0, -1000)
n_2 = Neuron(-65, -50, 20, 0, 0, -1000)

for t in range(1, T):
    if n_0_fired == True:
        n_1.I = w01 + stim[1][t]
        n_0_fired = False
    else:
        n_1.I = stim[1][t]

    if n_1_fired == True:
        n_2.I = w12 + stim[2][t]
        n_1_fired = False
    else:
        n_2.I = stim[2][t]
        
    
    if n_0.ref > 0:
        n_0.ref -= 1
    else:
        n_0.I = stim[0][t]
        n_0.status[t] = n_0.status[t-1] + dt/n_0.RC * (-(n_0.status[t-1] - n_0.V_rest) + n_0.I)
        if n_0.status[t] > n_0.V_thresh: # n_0 fired
            n_0.status[t] = 0 # visualization trick
            n_0.last_spike_time = t # record the timestamp
            # n_0.status[t] = n_0.V_rest # after the neuron has fired, reset it
            n_0.ref = 30 # refractory period
            n_0_fired = True
            dt01_stdp = n_1.last_spike_time - n_0.last_spike_time # calculate the timing
            if dt01_stdp < 0: # post before pre
                w01 -= A_minus * np.exp(dt01_stdp / tau_stdp) # update the input from n_0 to n_1
            
    if n_1.ref > 0:
        n_1.ref -= 1
    else:
        n_1.status[t] = n_1.status[t-1] + dt/n_1.RC * (-(n_1.status[t-1] - n_1.V_rest) + n_1.I)
        if n_1.status[t] > n_1.V_thresh: # n_1 fired
            n1_spike_times.append(t)
            n_1.status[t] = 0 # visualization trick
            # print(f"n_1 fired at t={t}") # it's tricky to see it reflected on the graph since firing and resetting happen simultaneously  
            n_1.last_spike_time = t
            # n_1.status[t] = n_1.V_rest
            n_1.ref = 30
            n_1_fired = True
            dt01_stdp = n_1.last_spike_time - n_0.last_spike_time
            if dt01_stdp > 0:  # pre before post
                w01 += A_plus * np.exp(-dt01_stdp / tau_stdp)
            dt12_stdp = n_2.last_spike_time - n_1.last_spike_time
            if dt12_stdp < 0: # post before pre
                w12 -= A_minus * np.exp(dt12_stdp / tau_stdp) # update the input from n_1 to n_2
    
    if n_2.ref > 0:
        n_2.ref -= 1
    else:
        n_2.status[t] = n_2.status[t-1] + dt/n_2.RC * (-(n_2.status[t-1] - n_2.V_rest) + n_2.I)
        if n_2.status[t] > n_2.V_thresh:
            n_2.status[t] = 0 # visualization trick
            # print(f"n_2 fired at t={t}") # it's tricky to see it reflected on the graph since firing and resetting happen simultaneously  
            n_2.last_spike_time = t
            # n_2.status[t] = n_2.V_rest
            n_2.ref = 30
            dt12_stdp = n_2.last_spike_time - n_1.last_spike_time
            if dt12_stdp > 0:  # pre before post
                w12 += A_plus * np.exp(-dt12_stdp / tau_stdp)


    w01_list[t] = w01
    w12_list[t] = w12

    
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(12, 8))
ax1.plot(n_0.status)
ax1.set_title('E1')
ax2.plot(n_1.status)
ax2.set_title('E2')
ax3.plot(n_2.status)
ax3.set_title('E3')
ax4.plot(w01_list, label='w01')
ax4.plot(w12_list, label='w12')
ax4.legend()
ax4.set_title('Weights')
plt.tight_layout()
plt.show()
for i in range(len(n1_spike_times)-1):
    print(n1_spike_times[i+1] - n1_spike_times[i], end=",")

print([t % 60 for t in n1_spike_times])
